In [21]:
topics = [
    "Which stocks did I talk to investing in with others",
    "What events did I plan to watch together with my friends", 
    "Did I play any games with other people recently? What games and which people?",
    "What homework projects am I doing at my Data Science University, who do I talk to about it?",
    "Discussing grades on recent exams and projects",
    "Talking about university assignments and issues with them"
]

In [8]:
from ollama_utils import get_vector_store
embedding_model_args={'model_name' : 'Qwen3-Embedding-4B-Q4KM:latest'}
vector_store = get_vector_store(**embedding_model_args)

In [ ]:
fetched_content = {}
for topic in topics[:1]:
    fetched_content[topic] = vector_store.similarity_search(query=topic,k=3)

In [25]:
for content in fetched_content[topic]:
    print(f'{content}\n\n---------------------')

page_content='**Metadata of Conversation**
        Chat: 🎁 (part 9) | Date: 2025-05-21 - 2025-05-21 | Language: ro
        **End of Metadata of Conversation** 
 Vasile Mereuță: vasile1mereuta voted for "Smh else" in the poll.
Dan: Chat gpt ultra pe jum de an
Dina🐝: Hey. Please vote, so we know how to proceed
Lore: Hm
Dan: Hai sal bagam si sal intrebam
Lore: Il fortam sa faca aici wishlistul apoi il scoatem din grup
horia: Ahm
𝙲𝚘𝚜𝚖𝚒𝚗𝚊: Are apple watch?
Dan: Vreo 3
Sab: Mie imi place idea de ps da nul stiu pe sandu asa de tare si nu chiar pot sa ma dau la parere
Dan: E big boss
horia: Da e milionar
eva: eva.tulea voted for "Yes" in the poll.
Dina🐝: Ultra are🤣
horia: atunci sa si ia el
horia: xbox
Lore: Poate ar vrea casti noi?
horia: gata am rezolvat
Lore: Din alea headphones
horia: horiaionescu__ changed the theme to Selena Gomez & Benny Blanco
Dina🐝: Dar serios. Daca aveti alte idei de cadouri. Va rog spuneti. Ca din partea mea, eu habar nam
Dina🐝: Dieta
Dina🐝: E complicat barbatubista

#### Load Reranking model

In [4]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig

device = 'mps'

def format_instructions(query, doc,instruction=None):
    # if instruction is None:
    #     instruction = "Given a question about specific details from chats of users, retrieve relevant chat passages that answer the query."
    output = "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}".format(instruction=instruction,query=query, doc=doc)
    return output

def process_inputs(pairs):
    inputs = tokenizer(
        pairs , padding=False, truncation = 'longest_first', ## longest first means it will shorten from the context chunk, not the query
        return_attention_mask = False, max_length = max_length - len(prefix_tokens) - len(suffix_tokens)
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = prefix_tokens + ele + suffix_tokens
    
    inputs = tokenizer.pad(inputs, padding=True,return_tensors='pt')
    for key in inputs:
        inputs[key] = inputs[key].to(device)
    return inputs

@torch.no_grad()
def compute_logits(inputs, **kwargs):
    batch_scores = model(**inputs).logits[:,-1,:]
    true_vector = batch_scores[:, token_true_id]
    false_vector = batch_scores[:, token_false_id]

    batch_scores = torch.stack([false_vector,true_vector],dim=1)
    batch_scores = torch.nn.functional.log_softmax(batch_scores,dim=1)
    scores = batch_scores[:,1].exp().tolist()

    return scores


repo_id = 'Qwen/Qwen3-Reranker-0.6B'
model = AutoModelForCausalLM.from_pretrained(repo_id,torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(repo_id)

model = model.to(device)

token_false_id = tokenizer.convert_tokens_to_ids("no")
token_true_id = tokenizer.convert_tokens_to_ids("yes")
max_length = 2048
prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)


In [4]:

task = 'Given a web search query, retrieve relevant passages that answer the query'

queries = ["What is the capital of China?",
    "Explain gravity",
]

documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

pairs = [format_instructions(instruction=task,query=query,doc=doc) for query,doc in zip(queries,documents)]
inputs = process_inputs(pairs)
scores = compute_logits(inputs)

print(f'scores: {scores}')

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/Users/pariidan/.pyenv/versions/3.12.0/envs/rag_langchain/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2695: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


scores: [0.91015625, 0.99951171875]


In [37]:
result = {}
fetched_content = {}
instruction = "Given a question about specific details from chats of users, retrieve relevant chat passages that answer the query or are at least relevant to it."
for topic in [topics[2]]:
   fetched_content[topic] = vector_store.similarity_search(query=topic,k=16)
   pairs = [format_instructions(instruction=instruction,query=topic,doc=doc.page_content) for doc in fetched_content[topic]]
   ids = [doc.id for doc in fetched_content[topic]]
   inputs = process_inputs(pairs)
   scores = compute_logits(inputs)
   
   # Create dictionary with topic -> {id: score} ordered by score descending
   id_score_pairs = sorted(zip(ids, scores), key=lambda x: x[1], reverse=True)
   result[topic] = {id_: score for id_, score in id_score_pairs}

In [32]:
import chromadb

## Add other_person name for pre-filtering chats in Chroma
client = chromadb.PersistentClient('.chroma_db')
collection_ig = client.get_collection('chat_documents_instagram_Qwen3-Embedding-0.6B-Q8_0-latest')
all_docs = collection_ig.get(include=["metadatas",'documents'])

In [36]:
print(collection_ig.get(ids='nido sorbo(2025-03-29 - 2025-04-11_11)')['documents'][0])

### Quantized Model for higher batch sizes

In [1]:
from utils.ollama_utils import get_vector_store
vector_store = get_vector_store(model_name='Qwen3-Embedding-0.6B-Q8_0:latest')

In [2]:
topic = "What plans did discuss of going on vacation to the beach?"
chunks = vector_store.similarity_search(query=topic,k=64)

In [3]:
from reranker import Reranker
qwen_reranker = Reranker(llama_cpp=True)
# #top_chunks = qwen_reranker.batch_rank_chunks(query=topic,chunks=chunks,batch_size=16,top_k=3)['chunks']

In [4]:
top_chunks = qwen_reranker.batch_rank_chunks(query=topic,chunks=chunks,batch_size=16,top_k=3)

In [ ]:
top_chunks = qwen_reranker.batch_rank_chunks(query=topic,chunks=chunks,batch_size=16,top_k=3)['chunks']

In [9]:
def format_instructions(instruction,query,doc):
    return "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}".format(
            instruction=instruction, query=query, doc=doc
        )

In [21]:
instruction = "Given a question about specific details from chat excerpts, retrieve relevant passages that answer the query."
#query = "Which stocks did I talk to investing in with others"
query = "What plans did discuss of going on vacation to the beach?"
pairs = [format_instructions(instruction=instruction, query=query, doc=doc.page_content) for doc in chunks]


In [ ]:
### llama.cpp test ( this gets me the log probs on ollama level speed)
from llama_cpp import Llama

llm = Llama(
      model_path="/Users/pariidan/Documents/python_projects/RAG_Langchain/models/Qwen3-Reranker-0.6B-q4_k_m.gguf",
      n_gpu_layers=-1, # Uncomment to use GPU acceleration
      logits_all=True,
      seed=1337, # Uncomment to set a specific seed
      n_ctx=4096 # Uncomment to increase the context window
)

In [34]:
prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
inputs = []
for pair in pairs[:64]:
    input = prefix + pair+ suffix
    inputs.append(input)

In [ ]:
import torch

input_probs = {}
for i, input in enumerate(inputs):
    # Initialize the dictionary entry first
    query_key = 'query' + str(i)
    input_probs[query_key] = {}
    
    output = llm(
        input,  # Prompt
        max_tokens=1,  # Generate up to 1 token
        echo=False,  # Don't echo the prompt back
        logprobs=True  # Return log probabilities
    )
    
    # Store the generated text
    input_probs[query_key]['decision'] = output['choices'][0]['text']
    
    # Extract and convert log probability to probability
    log_prob = torch.tensor(output['choices'][0]['logprobs']['token_logprobs'][0])
    prob = torch.exp(log_prob)
    input_probs[query_key]['prob'] = prob

In [ ]:
#### 1. Great, now need to get Yes and yes, combine probs select the highest ( if input does not have yes/no assume corrupt say no).abs
#### 2. Still should look into the faux batch generation it may offer

{'query0': {'decision': 'Yes', 'prob': tensor(0.0557)},
 'query1': {'decision': 'yes', 'prob': tensor(0.5442)},
 'query2': {'decision': 'no', 'prob': tensor(0.5017)},
 'query3': {'decision': 'no', 'prob': tensor(0.3647)},
 'query4': {'decision': 'No', 'prob': tensor(0.5083)},
 'query5': {'decision': 'no', 'prob': tensor(0.1827)},
 'query6': {'decision': 'No', 'prob': tensor(0.2916)},
 'query7': {'decision': 'no', 'prob': tensor(0.3663)},
 'query8': {'decision': 'no', 'prob': tensor(0.4518)},
 'query9': {'decision': 'No', 'prob': tensor(0.2431)},
 'query10': {'decision': 'yes', 'prob': tensor(0.2680)},
 'query11': {'decision': 'No', 'prob': tensor(0.4778)},
 'query12': {'decision': 'no', 'prob': tensor(0.4439)},
 'query13': {'decision': 'No', 'prob': tensor(0.2439)},
 'query14': {'decision': 'no', 'prob': tensor(0.1805)},
 'query15': {'decision': 'no', 'prob': tensor(0.5258)}}

#### Attempt with the LLama.cpp server

Running the llmaa.cpp model : 
        cd /opt/homebrew/bin

        ./llama-server \
        --model /Users/pariidan/Documents/python_projects/RAG_Langchain/models/Qwen3-Reranker-0.6B-q4_k_m.gguf \
        --port 10000 \
        -- host 0.0.0.0 \
        --ctx-size 4096 \
        --cont-batching  

In [12]:
import requests

URL = "http://localhost:10000/completion"

#### Synchronous

In [ ]:
for i, input in enumerate(inputs): 

    payload = {
        "prompt": inputs,
        "n_predict": 1,
        "n_probs": 3,   # ask for top-5 token probabilities
        "temperature": 0,
        "stream": False  # easier parsing, otherwise you'll need to handle streaming
    }

    resp = requests.post(URL, json=payload)

    if resp.ok:
        data = resp.json()
        print(f' {i}. {data[0]['content']}')
        #print("Generated text:", data["content"][0]["text"])
        
    else:
        print("Error:", resp.status_code, resp.text)

In [25]:
inputs

['<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n<Instruct>: Given a question about specific details from chat excerpts, retrieve relevant passages that answer the query.\n<Query>: What plans did discuss of going on vacation to the beach?\n<Document>: **Metadata of Conversation**\n            Chat: Mare August Grecia (part 39) | Date: 2024-08-03 - 2024-08-03 | Language: ro\n            **End of Metadata of Conversation** \n Cristian Ivanenco: Nu cred\nPuișor♥️🐢: dar e pzdt aglomerat cu feribot boi?\nPuișor♥️🐢: nu cu 16 da cu 14\nPuișor♥️🐢: 56 ca daca suntem grup merge skidka\nPuișor♥️🐢: zbsi\nPuișor♥️🐢: asa ca posibil 14 x 4 = 56 + 80 = 136\n\u202fCristian Ivanenco: 64\n\u202fCristian Ivanenco: Iese ca 4 zile\n\u202fCristian Ivanenco: Da\nPuișor♥️🐢: dar a 5 zi hz daca vom merge la plaja ca nu vom avea unde dus de facut\nPuișor♥️🐢: este de gru

#### Asynchronous

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

URL = "http://localhost:10000/completion"

def query(prompt, i):
    payload = {
        "prompt": prompt,
        "n_predict": 1,
        "n_probs": 3,
        "temperature": 0,
        "stream": False
    }
    resp = requests.post(URL, json=payload)
    if resp.ok:
        data = resp.json()
        return i, data
    else:
        return i, f"Error {resp.status_code}"

# Store results with their indices to preserve order
results = [None] * len(inputs)

with ThreadPoolExecutor(max_workers=len(inputs)) as executor:
    futures = [executor.submit(query, prompt, i) for i, prompt in enumerate(inputs)]
    
    for f in as_completed(futures):
        idx, result = f.result()
        results[idx] = result  # Store at correct index
        
        if isinstance(result, dict):  # Check if it's a successful response
            print(f"{idx}. Token: {result['content']}")
            print(f"Probability: {torch.exp(torch.tensor(result['completion_probabilities'][0]['logprob']))}"f)
            print('-----------------')

4. Token: No
Probability: 0.3920022249221802
-----------------
0. Token: yes
Probability: 0.5876863598823547
-----------------
9. Token: no
Probability: 0.3977030813694
-----------------
10. Token: No
Probability: 0.5070641040802002
-----------------
14. Token: no
Probability: 0.4129197597503662
-----------------
13. Token: No
Probability: 0.7177507877349854
-----------------
11. Token: no
Probability: 0.4607473611831665
-----------------
15. Token: No
Probability: 0.5849428176879883
-----------------
12. Token: No
Probability: 0.712431013584137
-----------------
17. Token: no
Probability: 0.4724932909011841
-----------------
19. Token: No
Probability: 0.39188799262046814
-----------------
16. Token: No
Probability: 0.36222153902053833
-----------------
21. Token: No
Probability: 0.647684633731842
-----------------
18. Token: no
Probability: 0.364477276802063
-----------------
20. Token: No
Probability: 0.6968426704406738
-----------------
22. Token: no
Probability: 0.31134888529777527

#### Async with compact post-processing

In [35]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

URL = "http://localhost:10000/completion"
k = 5

def query(prompt, i):
    resp = requests.post(URL, json={"prompt": prompt, "n_predict": 1, "n_probs": 3, "temperature": 0, "stream": False})
    if resp.ok:
        data = resp.json()
        return i, data['content'].strip().lower(), torch.exp(torch.tensor(data['completion_probabilities'][0]['logprob'])).item()
    return None

results = []
with ThreadPoolExecutor(max_workers=len(inputs)) as executor:
    for f in as_completed([executor.submit(query, prompt, i) for i, prompt in enumerate(inputs)]):
        if (r := f.result()) is not None:
            results.append(r)

yes = [(i, p) for i, c, p in results if c == 'yes']
final = sorted(yes, key=lambda x: x[1], reverse=True)[:k] if yes else sorted([(i, p) for i, c, p in results if c == 'no'], key=lambda x: x[1])[:k]

for i, p in final:
    print(f"{i}. Probability: {p:.4f}")

0. Probability: 0.5877


In [36]:
final

[(0, 0.5876863598823547)]

In [31]:
results[0]

{'index': 0,
 'content': 'yes',
 'tokens': [],
 'id_slot': 0,
 'stop': True,
 'model': 'gpt-3.5-turbo',
 'tokens_predicted': 1,
 'tokens_evaluated': 616,
 'generation_settings': {'n_predict': 1,
  'seed': 4294967295,
  'temperature': 0.0,
  'dynatemp_range': 0.0,
  'dynatemp_exponent': 1.0,
  'top_k': 40,
  'top_p': 0.949999988079071,
  'min_p': 0.05000000074505806,
  'top_n_sigma': -1.0,
  'xtc_probability': 0.0,
  'xtc_threshold': 0.10000000149011612,
  'typical_p': 1.0,
  'repeat_last_n': 64,
  'repeat_penalty': 1.0,
  'presence_penalty': 0.0,
  'frequency_penalty': 0.0,
  'dry_multiplier': 0.0,
  'dry_base': 1.75,
  'dry_allowed_length': 2,
  'dry_penalty_last_n': 4096,
  'dry_sequence_breakers': ['\n', ':', '"', '*'],
  'mirostat': 0,
  'mirostat_tau': 5.0,
  'mirostat_eta': 0.10000000149011612,
  'stop': [],
  'max_tokens': 1,
  'n_keep': 0,
  'n_discard': 0,
  'ignore_eos': False,
  'stream': False,
  'logit_bias': [],
  'n_probs': 3,
  'min_keep': 0,
  'grammar': '',
  'grammar

In [16]:
result["content"]

'no'